<a href="https://colab.research.google.com/github/leonardoddantas/llm-finetuning-rag-system/blob/main/01_rag_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Gerador de Dataset para Fine-Tuning (Instrução + Resposta)

Este notebook cria um dataset no formato `question` / `answer` a partir de documentos `.pdf` ou `.txt`.

**Objetivo:** gerar pares de pergunta curta + resposta direta com base no conteúdo do documento, usando um modelo de linguagem local (Qwen2.5-1.5B-Instruct).

**Saída:** arquivo JSONL (uma linha por exemplo), pronto para treinar ou ajustar um modelo RAG ou de QA.


## 1. Execução

Agora basta informar o caminho do seu arquivo (PDF ou TXT) e, opcionalmente, o número máximo de chunks para testar.

> **Dica:** comece com `max_chunks=5` para verificar se a geração está funcionando. Depois remova o parâmetro ou aumente o valor para processar o documento inteiro.

O modelo será baixado na primeira execução (cerca de 3 GB). Aguarde o download completo.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

In [1]:
!git clone https://github.com/leonardoddantas/llm-finetuning-rag-system.git

Cloning into 'llm-finetuning-rag-system'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 75 (delta 35), reused 38 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 4.70 MiB | 11.10 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [2]:
%cd /content/llm-finetuning-rag-system

import sys
sys.path.append("/content/llm-finetuning-rag-system")

/content/llm-finetuning-rag-system


In [ ]:
!mkdir -p data/processed

In [3]:
!pip install transformers torch accelerate pdfplumber tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 123.6 MB/s eta 0:00:00


In [4]:
from src.dataset_generation.generator import generate_dataset

caminho_arquivo = "data/raw/20260045290.pdf"

generate_dataset(
    file_path=caminho_arquivo,
    model_id="microsoft/Phi-4-mini-instruct",
    output_file="data/processed/dataset_gerado.jsonl",
)

🔄 Carregando modelo: microsoft/Phi-4-mini-instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

📄 Extraindo texto do arquivo...
✂️  Dividindo em chunks...
🧠 Gerando pares (instrução + resposta) para 277 chunks...


Processando chunks: 100%|██████████| 277/277 [18:33<00:00,  4.02s/it]


✅ Dataset salvo em: data/processed/dataset_gerado.jsonl (272 exemplos gerados)


## 🔍 Observações finais

- O modelo utilizado é o **Qwen2.5-1.5B-Instruct**, que funciona bem em CPU ou GPU.
- Para documentos longos, o processo pode levar alguns minutos/horas, dependendo do hardware.
- O dataset gerado pode ser usado para fine-tuning de modelos menores (como BERT, Llama, etc.) ou como base para RAG.
- Se desejar modificar o comprimento das perguntas/respostas, ajuste o `prompt` na função `generate_instruction_response`.

**Divirta-se gerando dados!** 🚀